In [ ]:
import re
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import pandas as pd

# ----------------
# Inputs (edit if needed)
# ----------------
INPUT_FILES = [
    "registry_webnlg_en_ca.csv"
]

VOTE_THRESHOLD_DEFAULT = 0.90

# CSV schema (must match exactly)
CSV_COLUMNS = [
    "status","processed_at","xml_path","category","eid","source_lid","source_en",
    "ca_nllb","ca_madlad","ca_salamandra",
    "vote_threshold",
    "sim_nllb_madlad","sim_nllb_salamandra","sim_madlad_salamandra",
    "agree_nllb_madlad","agree_nllb_salamandra","agree_madlad_salamandra",
    "voted_ca","final_ca","error"
]

# ----------------
# Robust normalization
# ----------------
_ws_re = re.compile(r"\s+")

def _norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return _ws_re.sub(" ", str(s).strip())

def _sim(a, b) -> float:
    return SequenceMatcher(None, _norm(a).lower(), _norm(b).lower()).ratio()

def mojibake_penalty(text) -> float:
    # keep placeholder; return 0.0 so it doesn't affect scoring
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return 0.0
    return 0.0

def safe_thr(x) -> float:
    try:
        v = float(x)
        if v <= 0 or v > 1.0:
            return VOTE_THRESHOLD_DEFAULT
        return v
    except Exception:
        return VOTE_THRESHOLD_DEFAULT

# ----------------
# Voting: cluster-first + medoid
# ----------------
def vote_3way_medoid_cluster_first(t1, t2, t3, thr: float = 0.90) -> str:
    texts = {"nllb": _norm(t1), "madlad": _norm(t2), "salamandra": _norm(t3)}
    cands = ["nllb", "madlad", "salamandra"]

    def s(a, b):
        if a == b:
            return 1.0
        return _sim(texts[a], texts[b])

    edges = []
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            a, b = cands[i], cands[j]
            if s(a, b) >= thr:
                edges.append((a, b))

    adj = {c: set() for c in cands}
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)

    seen = set()
    comps = []
    for c in cands:
        if c in seen:
            continue
        stack = [c]
        comp = set()
        while stack:
            x = stack.pop()
            if x in seen:
                continue
            seen.add(x)
            comp.add(x)
            stack.extend(adj[x])
        comps.append(comp)

    comps.sort(key=lambda comp: (-len(comp), sorted(comp)))
    best_comp = comps[0]

    def score(c):
        return (s(c, "nllb") + s(c, "madlad") + s(c, "salamandra") - 1.0) - mojibake_penalty(texts[c])

    candidates = list(best_comp) if len(best_comp) == 2 else cands
    winner = max(candidates, key=score)
    return texts[winner]

def vote_details(t1, t2, t3, threshold: float):
    s12 = _sim(t1, t2)
    s13 = _sim(t1, t3)
    s23 = _sim(t2, t3)
    return {
        "sim_nllb_madlad": float(f"{s12:.4f}"),
        "sim_nllb_salamandra": float(f"{s13:.4f}"),
        "sim_madlad_salamandra": float(f"{s23:.4f}"),
        "agree_nllb_madlad": int(s12 >= threshold),
        "agree_nllb_salamandra": int(s13 >= threshold),
        "agree_madlad_salamandra": int(s23 >= threshold),
    }

# ----------------
# Re-vote one registry (CA-only, fixed columns)
# ----------------
def revote_verbalisation_registry(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Drop rows with status != "OK" (also handles missing)
    if "status" in df.columns:
        df = df[df["status"].astype(str).eq("OK")].copy()
    else:
        # If status is missing, treat all as not OK -> empty result
        df = df.iloc[0:0].copy()

    # 2) Ensure all required columns exist (create missing as NA)
    for c in CSV_COLUMNS:
        if c not in df.columns:
            df[c] = pd.NA

    # 3) Reorder to exact schema, dropping any extra columns
    df = df[CSV_COLUMNS].copy()

    # 4) Vote using CA columns
    c_nllb = "ca_nllb"
    c_mad  = "ca_madlad"
    c_sal  = "ca_salamandra"
    c_voted = "voted_ca"
    c_final = "final_ca"

    for i, row in df.iterrows():
        t1 = row.get(c_nllb, "")
        t2 = row.get(c_mad, "")
        t3 = row.get(c_sal, "")
        thr = safe_thr(row.get("vote_threshold", VOTE_THRESHOLD_DEFAULT))

        det = vote_details(t1, t2, t3, thr)
        for k, v in det.items():
            df.at[i, k] = v

        voted = vote_3way_medoid_cluster_first(t1, t2, t3, thr=thr)
        df.at[i, c_voted] = voted
        df.at[i, c_final] = voted
        df.at[i, "vote_threshold"] = thr
        df.at[i, "processed_at"] = datetime.now().isoformat(timespec="seconds")
        df.at[i, "error"] = pd.NA  # clear error on reprocess (optional)

    return df

# ----------------
# Run all
# ----------------
for fp in INPUT_FILES:
    in_path = Path(fp)
    df = pd.read_csv(in_path)

    out_df = revote_verbalisation_registry(df)

    out_path = in_path.with_suffix("")  # remove .csv
    out_path = Path(str(out_path) + ".revoted.csv")
    out_df.to_csv(out_path, index=False)

    # quick report
    changed = "n/a"
    if "final_ca" in df.columns and len(out_df) > 0:
        # compare against original finals for the subset that survived status==OK
        orig_ok = df[df.get("status", pd.Series([], dtype=str)).astype(str).eq("OK")].copy()
        if "final_ca" in orig_ok.columns and len(orig_ok) == len(out_df):
            changed = (out_df["final_ca"].astype(str).values != orig_ok["final_ca"].astype(str).values).sum()
    print(f"{fp} -> {out_path.name} | kept_rows={len(out_df)} | changed finals: {changed}")